# Weights Manager

This notebook is dedicated to weight management only.

Workflow:
1. List all saved Base/CBAM weights and temp run candidates.
2. Edit labels/notes for saved weights.
3. Select one active weight for training fine-tune or inference.
4. After training, choose a temp candidate and promote (save) or discard.

In [1]:
import json
import shutil
from datetime import datetime
from pathlib import Path

WEIGHTS_ROOT = Path('../models/weights')
BASE_DIR = WEIGHTS_ROOT / 'Base'
CBAM_DIR = WEIGHTS_ROOT / 'CBAM'
RUNS_TMP_DIR = WEIGHTS_ROOT / '_runs_tmp'
REGISTRY_PATH = WEIGHTS_ROOT / 'weights_registry.json'
ACTIVE_SELECTION_PATH = WEIGHTS_ROOT / 'active_weight_selection.json'
LAST_TRAINING_CANDIDATE_PATH = WEIGHTS_ROOT / 'last_training_candidate.json'
LEGACY_ACTIVE_PATH = WEIGHTS_ROOT / 'HRNet.pth'

for folder in [WEIGHTS_ROOT, BASE_DIR, CBAM_DIR, RUNS_TMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def _load_registry():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH, 'r') as f:
            return json.load(f)
    return {'weights': []}


def _save_registry(data):
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(data, f, indent=2)


def _load_sidecar(weight_path: Path):
    sidecar = weight_path.with_suffix('.json')
    if sidecar.exists():
        try:
            with open(sidecar, 'r') as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def _collect_library():
    rows = []
    for family, folder in [('Base', BASE_DIR), ('CBAM', CBAM_DIR)]:
        for p in sorted(folder.glob('*.pth'), key=lambda x: x.stat().st_mtime, reverse=True):
            meta = _load_sidecar(p)
            rows.append({
                'family': family,
                'name': p.stem,
                'path': str(p.resolve()),
                'size_mb': p.stat().st_size / 1e6,
                'mtime': datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
                'best_score': meta.get('best_score', '-'),
                'notes': meta.get('notes', '')
            })
    return rows


def _collect_temp_candidates():
    """
    Detect temp candidates in two forms:
    1) Structured runs: _runs_tmp/<run_name>/best.pth
    2) Loose files: any .pth under _runs_tmp (including nested folders)
    """
    rows = []
    seen_paths = set()

    # Structured run format: .../<run_name>/best.pth
    structured_best_files = sorted(
        RUNS_TMP_DIR.rglob('best.pth'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    for best_path in structured_best_files:
        run_dir = best_path.parent
        meta_path = run_dir / 'training_metadata.json'
        meta = {}
        if meta_path.exists():
            try:
                with open(meta_path, 'r') as f:
                    meta = json.load(f)
            except Exception:
                meta = {}

        rows.append({
            'candidate_type': 'structured',
            'run_name': run_dir.name,
            'run_dir': str(run_dir.resolve()),
            'best_path': str(best_path.resolve()),
            'meta_path': str(meta_path.resolve()) if meta_path.exists() else '',
            'best_loss': meta.get('best_loss', None),
            'last_saved_epoch': meta.get('last_saved_epoch', None),
            'mtime': datetime.fromtimestamp(best_path.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        })
        seen_paths.add(str(best_path.resolve()).lower())

    # Loose/nested .pth files under _runs_tmp
    loose_files = sorted(
        RUNS_TMP_DIR.rglob('*.pth'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    for p in loose_files:
        p_resolved = str(p.resolve()).lower()
        if p_resolved in seen_paths:
            continue

        # Try to find metadata near the file
        meta_path = p.parent / 'training_metadata.json'
        if not meta_path.exists() and p.parent.name == 'checkpoints':
            meta_path = p.parent.parent / 'training_metadata.json'

        meta = {}
        if meta_path.exists():
            try:
                with open(meta_path, 'r') as f:
                    meta = json.load(f)
            except Exception:
                meta = {}

        rows.append({
            'candidate_type': 'loose',
            'run_name': f"{p.parent.name}/{p.stem}",
            'run_dir': str(p.parent.resolve()),
            'best_path': str(p.resolve()),
            'meta_path': str(meta_path.resolve()) if meta_path.exists() else '',
            'best_loss': meta.get('best_loss', None),
            'last_saved_epoch': meta.get('last_saved_epoch', None),
            'mtime': datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        })

    return rows


def list_all_weights():
    library = _collect_library()
    temp = _collect_temp_candidates()

    print('=' * 110)
    print('LIBRARY WEIGHTS (SAVED)')
    print('=' * 110)
    if not library:
        print('(empty)')
    for i, row in enumerate(library):
        print(f"{i:2d} | {row['family']:4s} | {row['name'][:35]:35s} | {str(row['best_score'])[:10]:10s} | {row['size_mb']:6.2f} MB | {row['mtime']}")
        if row['notes']:
            print(f"     notes: {row['notes']}")

    print('\n' + '=' * 110)
    print('TEMP RUN CANDIDATES (NOT PROMOTED YET)')
    print('=' * 110)
    if not temp:
        print('(empty)')
    for i, row in enumerate(temp):
        best_loss_txt = '-' if row['best_loss'] is None else f"{row['best_loss']:.6f}"
        ctype = row.get('candidate_type', 'unknown')
        print(f"{i:2d} | {ctype:10s} | {row['run_name'][:34]:34s} | best_loss={best_loss_txt:10s} | epoch={str(row['last_saved_epoch']):5s} | {row['mtime']}")

    if LAST_TRAINING_CANDIDATE_PATH.exists():
        try:
            with open(LAST_TRAINING_CANDIDATE_PATH, 'r') as f:
                latest = json.load(f)
            print('\nLatest training candidate summary:')
            print(json.dumps(latest, indent=2))
        except Exception as e:
            print(f'Could not read latest candidate summary: {e}')

    return library, temp


def set_active_selection(weight_path: Path, purpose='train_or_infer', also_copy_to_legacy=False):
    if not weight_path.exists():
        raise FileNotFoundError(f'Weight does not exist: {weight_path}')

    payload = {
        'path': str(weight_path.resolve()),
        'name': weight_path.stem,
        'purpose': purpose,
        'set_at': datetime.now().isoformat()
    }
    with open(ACTIVE_SELECTION_PATH, 'w') as f:
        json.dump(payload, f, indent=2)

    if also_copy_to_legacy:
        shutil.copy2(weight_path, LEGACY_ACTIVE_PATH)

    print(f'Active selection saved: {ACTIVE_SELECTION_PATH}')
    print(f'Selected weight: {weight_path}')


def promote_candidate(candidate_best_path: Path, family: str, label: str, notes: str, output_name: str = ''):
    if not candidate_best_path.exists():
        raise FileNotFoundError(f'Candidate weight missing: {candidate_best_path}')

    destination_dir = BASE_DIR if family == 'Base' else CBAM_DIR
    destination_dir.mkdir(parents=True, exist_ok=True)

    if output_name.strip():
        stem = output_name.strip().replace(' ', '_')
    else:
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        label_part = f'_{label}' if label else ''
        stem = f'{stamp}_{family.lower()}{label_part}'

    target = destination_dir / f'{stem}.pth'
    suffix = 2
    while target.exists():
        target = destination_dir / f'{stem}_v{suffix}.pth'
        suffix += 1

    shutil.copy2(candidate_best_path, target)

    sidecar = {
        'name': target.stem,
        'family': family,
        'best_score': label,
        'notes': notes,
        'source_candidate': str(candidate_best_path.resolve()),
        'created_at': datetime.now().isoformat()
    }
    with open(target.with_suffix('.json'), 'w') as f:
        json.dump(sidecar, f, indent=2)

    registry = _load_registry()
    registry['weights'] = [w for w in registry.get('weights', []) if w.get('path') != str(target.resolve())]
    registry['weights'].append({
        'name': target.stem,
        'family': family,
        'path': str(target.resolve()),
        'best_score': label,
        'notes': notes,
        'created_at': datetime.now().isoformat()
    })
    _save_registry(registry)

    return target


print('Weights manager helpers loaded.')

Weights manager helpers loaded.


## Cell 3: List All Saved and Temp Weights

In [2]:
library_rows, temp_rows = list_all_weights()

LIBRARY WEIGHTS (SAVED)
 0 | Base | 20260423_012204_base_Bilinear 3nd r | Bilinear 3 |   2.38 MB | 2026-04-23 01:16:23
 1 | Base | HRNet(30.64 just right)             | -          |   2.49 MB | 2026-04-20 02:38:58
 2 | Base | HRNet(29 too smooth)                | -          |   2.49 MB | 2026-04-20 02:03:39
 3 | Base | HRNet(30.58db too sharp)            | -          |   2.49 MB | 2026-04-20 00:05:50
 4 | CBAM | 20260423_235944_cbam_CBAM 1.5 bias  | CBAM 1.5 b |   2.39 MB | 2026-04-23 23:56:15
 5 | CBAM | 20260423_232605_base_CBAM new scrip | CBAM new s |   2.39 MB | 2026-04-23 23:23:50

TEMP RUN CANDIDATES (NOT PROMOTED YET)
 0 | structured | 20260423_234703_cbam_finetune_coil | best_loss=0.018315   | epoch=30    | 2026-04-23 23:56:15
 1 | loose      | checkpoints/checkpoint_epoch_25    | best_loss=0.018315   | epoch=30    | 2026-04-23 23:53:43

Latest training candidate summary:
{
  "run_name": "20260423_234703_cbam_finetune_coil_experiment",
  "run_dir": "..\\models\\weights\\_runs_

## Cell 5: Edit Existing Label/Notes

Use this on already saved library weights (Base/CBAM).

In [ ]:
# 1) Run Cell 4 first to refresh library_rows
# 2) Pick a row index from library_rows
EDIT_LIBRARY_INDEX = None  # example: 0
NEW_LABEL = ''             # example: '30.5dB'
NEW_NOTES = ''             # example: 'best circles so far'

if EDIT_LIBRARY_INDEX is None:
    print('Set EDIT_LIBRARY_INDEX and re-run this cell.')
else:
    if EDIT_LIBRARY_INDEX < 0 or EDIT_LIBRARY_INDEX >= len(library_rows):
        raise ValueError('Invalid EDIT_LIBRARY_INDEX')

    row = library_rows[EDIT_LIBRARY_INDEX]
    weight_path = Path(row['path'])
    sidecar_path = weight_path.with_suffix('.json')
    sidecar = _load_sidecar(weight_path)

    if NEW_LABEL != '':
        sidecar['best_score'] = NEW_LABEL
    if NEW_NOTES != '':
        sidecar['notes'] = NEW_NOTES

    sidecar.setdefault('name', weight_path.stem)
    sidecar.setdefault('family', row['family'])
    sidecar['updated_at'] = datetime.now().isoformat()

    with open(sidecar_path, 'w') as f:
        json.dump(sidecar, f, indent=2)

    registry = _load_registry()
    for rec in registry.get('weights', []):
        if rec.get('path') == str(weight_path.resolve()):
            if NEW_LABEL != '':
                rec['best_score'] = NEW_LABEL
            if NEW_NOTES != '':
                rec['notes'] = NEW_NOTES
            rec['updated_at'] = datetime.now().isoformat()
    _save_registry(registry)

    print(f'Updated metadata for: {weight_path}')

## Cell 7: Select and Load Active Weight

This sets a shared active selection file that training/inference notebooks can read.

In [3]:
# 1) Run Cell 4 to refresh library_rows
# 2) Choose index and set options
SELECT_LIBRARY_INDEX = 4  # set explicitly, e.g. 0
SELECTION_NAME_CONTAINS = ''  # optional safer selector, e.g. '30.64'
SELECTION_PURPOSE = 'train_or_infer'
COPY_TO_LEGACY_HRNET = False  # True copies to models/weights/HRNet.pth

if SELECT_LIBRARY_INDEX is None and not SELECTION_NAME_CONTAINS.strip():
    print('Set SELECT_LIBRARY_INDEX or SELECTION_NAME_CONTAINS and re-run this cell.')
else:
    if SELECTION_NAME_CONTAINS.strip():
        query = SELECTION_NAME_CONTAINS.strip().lower()
        matches = [
            i for i, row in enumerate(library_rows)
            if query in row['name'].lower() or query in row['path'].lower()
        ]
        if len(matches) == 0:
            raise ValueError(f'No library weight matched query: {SELECTION_NAME_CONTAINS}')
        if len(matches) > 1:
            raise ValueError(f'Ambiguous query, multiple matches at indices: {matches}')
        SELECT_LIBRARY_INDEX = matches[0]

    if SELECT_LIBRARY_INDEX < 0 or SELECT_LIBRARY_INDEX >= len(library_rows):
        raise ValueError('Invalid SELECT_LIBRARY_INDEX')

    selected = library_rows[SELECT_LIBRARY_INDEX]
    print('Selecting active weight:')
    print(f"  index: {SELECT_LIBRARY_INDEX}")
    print(f"  family: {selected['family']}")
    print(f"  name: {selected['name']}")
    print(f"  path: {selected['path']}")

    set_active_selection(
        weight_path=Path(selected['path']),
        purpose=SELECTION_PURPOSE,
        also_copy_to_legacy=COPY_TO_LEGACY_HRNET
    )

Selecting active weight:
  index: 4
  family: CBAM
  name: 20260423_235944_cbam_CBAM 1.5 bias
  path: D:\GUC\HighResNet model\HighRes-net\models\weights\CBAM\20260423_235944_cbam_CBAM 1.5 bias.pth
Active selection saved: ..\models\weights\active_weight_selection.json
Selected weight: D:\GUC\HighResNet model\HighRes-net\models\weights\CBAM\20260423_235944_cbam_CBAM 1.5 bias.pth


## Cell 9: Post-Training Save or Discard

Promote a temp-run candidate to Base/CBAM after reviewing training plots and diagnostics.

In [3]:
# 1) Run Cell 4 first to refresh temp_rows
# 2) Choose a temp candidate index
PROMOTE_TEMP_INDEX = 0        # set explicitly, e.g. 0
POST_KEEP_RESULT = True          # True=save/promote, False=discard, None=no action
POST_DEST_FAMILY = 'CBAM'        # 'Base' or 'CBAM'
POST_LABEL = 'CBAM 1.5 bias'   # optional label, e.g. '30.5dB'
POST_NOTES = ''                  # optional notes
POST_OUTPUT_NAME = ''            # optional full custom file stem
DELETE_TEMP_RUN_ON_DISCARD = True

if PROMOTE_TEMP_INDEX is None:
    print('Set PROMOTE_TEMP_INDEX and re-run this cell.')
elif POST_KEEP_RESULT is None:
    print('Set POST_KEEP_RESULT=True or False and re-run this cell.')
else:
    if PROMOTE_TEMP_INDEX < 0 or PROMOTE_TEMP_INDEX >= len(temp_rows):
        raise ValueError('Invalid PROMOTE_TEMP_INDEX')

    candidate = temp_rows[PROMOTE_TEMP_INDEX]
    candidate_best = Path(candidate['best_path'])
    candidate_run_dir = Path(candidate['run_dir'])

    if POST_KEEP_RESULT:
        saved_path = promote_candidate(
            candidate_best_path=candidate_best,
            family=POST_DEST_FAMILY,
            label=POST_LABEL,
            notes=POST_NOTES,
            output_name=POST_OUTPUT_NAME,
        )
        print(f'Promoted to library: {saved_path}')
    else:
        print('Candidate discarded (not promoted).')
        if DELETE_TEMP_RUN_ON_DISCARD and candidate_run_dir.exists():
            shutil.rmtree(candidate_run_dir)
            print(f'Deleted temp run directory: {candidate_run_dir}')

    print('\nRefresh listing:')
    library_rows, temp_rows = list_all_weights()

Promoted to library: ..\models\weights\CBAM\20260423_235944_cbam_CBAM 1.5 bias.pth

Refresh listing:
LIBRARY WEIGHTS (SAVED)
 0 | Base | 20260423_012204_base_Bilinear 3nd r | Bilinear 3 |   2.38 MB | 2026-04-23 01:16:23
 1 | Base | HRNet(30.64 just right)             | -          |   2.49 MB | 2026-04-20 02:38:58
 2 | Base | HRNet(29 too smooth)                | -          |   2.49 MB | 2026-04-20 02:03:39
 3 | Base | HRNet(30.58db too sharp)            | -          |   2.49 MB | 2026-04-20 00:05:50
 4 | CBAM | 20260423_235944_cbam_CBAM 1.5 bias  | CBAM 1.5 b |   2.39 MB | 2026-04-23 23:56:15
 5 | CBAM | 20260423_232605_base_CBAM new scrip | CBAM new s |   2.39 MB | 2026-04-23 23:23:50

TEMP RUN CANDIDATES (NOT PROMOTED YET)
 0 | structured | 20260423_234703_cbam_finetune_coil | best_loss=0.018315   | epoch=30    | 2026-04-23 23:56:15
 1 | loose      | checkpoints/checkpoint_epoch_25    | best_loss=0.018315   | epoch=30    | 2026-04-23 23:53:43

Latest training candidate summary:
{
  "